In [1]:
print("Kernel avviata con successo")

Kernel avviata con successo


In [2]:
# CELL 2: Visual Feature Extraction with CLIP
import os
import cv2
import torch
import numpy as np
from PIL import Image
from tqdm.notebook import tqdm
from transformers import CLIPProcessor, CLIPModel
import torch
print(torch.__version__) 

2.6.0+cu124


In [ ]:
#!pip uninstall torch torchvision torchaudio -y
#!pip install torch==2.6.0 torchvision==0.21.0 torchaudio==2.6.0 --index-url https://download.pytorch.org/whl/cu124
#!pip install --upgrade torch torchvision torchaudio
#!pip install ipywidgets
#!pip install resampy

In [3]:
PROJECT_PATH = "Thesis_Data"

# 1. Load CLIP
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Loading CLIP model on {device}...")
model_clip = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
processor_clip = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

def extract_clip_features(video_path):
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    if fps <= 0: fps = 30
    
    frame_features = []
    count = 0
    
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
            
        # Extract 1 frame per second
        if count % int(fps) == 0:
            # Convert BGR (OpenCV) to RGB (PIL)
            img = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            pil_img = Image.fromarray(img)
            
            inputs = processor_clip(images=pil_img, return_tensors="pt").to(device)
            
            with torch.no_grad():
                # --- THE BULLETPROOF FIX ---
                # 1. Run the vision part of the model explicitly (This outputs the "Object")
                vision_outputs = model_clip.vision_model(pixel_values=inputs['pixel_values'])
                
                # 2. Extract the raw tensor from inside that Object
                pooled_tensor = vision_outputs.pooler_output 
                
                # 3. Project it into the final 512-dimensional CLIP space
                features = model_clip.visual_projection(pooled_tensor)
                
            frame_features.append(features.cpu().numpy().flatten())
        count += 1
        
    cap.release()
    
    if not frame_features:
        return np.zeros(512 * 4)  # 4x perché usiamo 4 statistiche
    
    arr = np.array(frame_features)
    mean = np.mean(arr, axis=0)
    std  = np.std(arr, axis=0)
    mx   = np.max(arr, axis=0)
    mn   = np.min(arr, axis=0)
    
    return np.concatenate([mean, std, mx, mn])  # → 2048 dim
        
    ## Average the features across the whole video
    #return np.mean(frame_features, axis=0)

# 2. Process all videos
visual_features_dict_clip = {}
video_files = [f for f in os.listdir(f"{PROJECT_PATH}/videos") if f.endswith('.mp4')]

Loading CLIP model on cuda...


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


In [ ]:
# CELL 2: Visual Feature Extraction with CLIP



print("Extracting CLIP features (Visual)...")
for v_file in tqdm(video_files):
    v_id = v_file.replace(".mp4", "")
    v_path = f"{PROJECT_PATH}/videos/{v_file}"
    try:
        visual_features_dict_clip[v_id] = extract_clip_features(v_path)
    except Exception as e:
        print(f"Error on {v_id}: {e}")

# 3. Save with a NEW name
np.save(f"{PROJECT_PATH}/visual_features_clip.npy", visual_features_dict_clip)
print("CLIP Visual features saved successfully!")

In [ ]:
# CELL 3: Audio Feature Extraction with VGGishm

# 1. Load VGGish from PyTorch Hub
print("Loading VGGish model...")
# We use harritaylor's implementation which is the standard PyTorch port for VGGish
vggish = torch.hub.load('harritaylor/torchvggish', 'vggish')
vggish.eval()

def extract_vggish_features(audio_path):
    # The harritaylor VGGish port takes a wav file path directly,
    # automatically resamples it to 16kHz, computes the mel-spectrogram, 
    # and runs it through the network!
    with torch.no_grad():
        # Outputs shape: [number_of_seconds, 128]
        embeddings = vggish.forward(audio_path)
    
    # Average the embeddings over time to get one 128-D vector for the whole ad
    # Convert tensor to numpy
    features_np = embeddings.cpu().numpy()
    
    if len(features_np) == 0:
        return np.zeros(128)
        
    return np.mean(features_np, axis=0)

# 2. Process all audio files
audio_features_dict_vggish = {}
audio_files = [f for f in os.listdir(f"{PROJECT_PATH}/audio") if f.endswith('.wav')]


In [ ]:

print("Extracting VGGish features (Audio)...")
for a_file in tqdm(audio_files):
    v_id = a_file.replace(".wav", "")
    a_path = f"{PROJECT_PATH}/audio/{a_file}"
    try:
        audio_features_dict_vggish[v_id] = extract_vggish_features(a_path)
    except Exception as e:
        print(f"Error on {v_id}: {e}")

# 3. Save with a NEW name
np.save(f"{PROJECT_PATH}/audio_features_vggish.npy", audio_features_dict_vggish)
print("VGGish Audio features saved successfully!")

In [4]:
# MASTER CELL: Data Loading, Deep Late Fusion, & Early Stopping
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, accuracy_score



In [5]:
# ==========================================
# 1. LOAD AND PREPARE DATA (Fixes the NameError)
# ==========================================
PROJECT_PATH = "Thesis_Data"
LABELS_PATH = "videos_with_sentiment_labels.csv"

df = pd.read_csv(LABELS_PATH)
visual_dict = np.load(f"{PROJECT_PATH}/visual_features_clip.npy", allow_pickle=True).item()
audio_dict = np.load(f"{PROJECT_PATH}/audio_features_vggish.npy", allow_pickle=True).item()

X_visual, X_audio, y_labels = [], [], []

for index, row in df.iterrows():
    v_id = row['video_id']
    label = row['majority_sentiment']
    if v_id in visual_dict and v_id in audio_dict:
        X_visual.append(visual_dict[v_id])
        X_audio.append(audio_dict[v_id])
        y_labels.append(label)

X_visual = np.array(X_visual)
X_audio = np.array(X_audio)
y_labels = np.array(y_labels)

le = LabelEncoder()
y_encoded = le.fit_transform(y_labels)



## Cross Attention in Fusione Intermedia

In [ ]:
import logging
import sys
from sklearn.metrics import f1_score


import torch
import torch.nn as nn


SHARED_DIM = 128 # 64 o 256 e divisibile per NUM_HEADS
NUM_HEADS = 2 # o 4 o 8
DROPOUT = 0.1 # 0.1 o 0.2 o 0.3
LEARNING_RATE = 3e-4 # 1e-3

# ==========================================
# MODULO 1: Cross-Modal Attention
# ==========================================
class CrossModalAttention(nn.Module):
    """
    Il visual branch usa l'audio come contesto tramite cross-attention.
    - Query  = visual features  (il modello 'chiede' informazioni all'audio)
    - Key/Value = audio features (l'audio 'risponde' con il suo contenuto)
    Il risultato è una versione delle feature visive arricchita dal contesto audio.
    """
    def __init__(self, dim=128, NUM_HEADS=2, DROPOUT=0.1):
        super().__init__()
        assert dim % NUM_HEADS == 0, "dim deve essere divisibile per NUM_HEADS"
        
        self.attn = nn.MultiheadAttention(
            embed_dim=dim,
            num_heads=NUM_HEADS,
            dropout=DROPOUT,
            batch_first=True   # input shape: (batch, seq, dim) — più intuitivo
        )
        self.norm = nn.LayerNorm(dim)
        self.Dropout = nn.Dropout(DROPOUT)



    def forward(self, visual_feats, audio_feats):
        """
        Args:
            visual_feats: (B, dim) — feature visive dopo l'encoder
            audio_feats:  (B, dim) — feature audio dopo l'encoder
        Returns:
            (B, dim) — visual features aggiornate con contesto audio
        """
        # MultiheadAttention si aspetta (B, seq_len, dim)
        # I nostri vettori sono già aggregati nel tempo → seq_len = 1
        q = visual_feats.unsqueeze(1)   # (B, 1, dim)
        k = audio_feats.unsqueeze(1)    # (B, 1, dim)
        v = audio_feats.unsqueeze(1)    # (B, 1, dim)

        attended, _ = self.attn(q, k, v)  # (B, 1, dim)
        attended = attended.squeeze(1)     # (B, dim)

        # Connessione residuale + normalizzazione
        out = self.norm(visual_feats + self.Dropout(attended))
        return out  # (B, dim)


# ==========================================
# MODULO 2: Rete con CrossModalAttention integrata
# ==========================================
class FusionWithCrossAttention(nn.Module):
    def __init__(self, visual_dim=512, audio_dim=128, SHARED_DIM=128, num_classes=3, NUM_HEADS=2, DROPOUT=0.1):
        super().__init__()

        # Encoder VISIVO
        self.visual_encoder = nn.Sequential(
            nn.Linear(visual_dim, 256),
            nn.LayerNorm(256), nn.GELU(), nn.Dropout(DROPOUT),
            nn.Linear(256, 256),
            nn.LayerNorm(256), nn.GELU(), nn.Dropout(DROPOUT),
            nn.Linear(256, SHARED_DIM),
            nn.LayerNorm(SHARED_DIM), nn.GELU()
        )

        # Encoder AUDIO
        self.audio_encoder = nn.Sequential(
            nn.Linear(audio_dim, 256),
            nn.LayerNorm(256), nn.GELU(), nn.Dropout(DROPOUT),
            nn.Linear(256, 256),
            nn.LayerNorm(256), nn.GELU(), nn.Dropout(DROPOUT),
            nn.Linear(256, SHARED_DIM),
            nn.LayerNorm(SHARED_DIM), nn.GELU()
        )

        # Cross Attention bidirezionale
        self.cross_attn = CrossModalAttention(dim=SHARED_DIM, NUM_HEADS=NUM_HEADS)
        self.cross_attn_a = CrossModalAttention(dim=SHARED_DIM, NUM_HEADS=NUM_HEADS)

        # ✅ GATING (ATTIVO)
        self.gate = nn.Sequential(
            nn.Linear(SHARED_DIM * 2, 2),   # ✅ CORRETTO
            nn.Softmax(dim=-1)
        )

        # Classifier (lasciato invariato)
        self.classifier = nn.Sequential(
            nn.Linear(SHARED_DIM, 128),
            nn.GELU(), nn.Dropout(DROPOUT),
            nn.Linear(128, 64),
            nn.GELU(),
            nn.Linear(64, num_classes)
        )

    def forward(self, visual_x, audio_x):
        v = self.visual_encoder(visual_x)
        a = self.audio_encoder(audio_x)

        # Cross attention
        v_attended = self.cross_attn(v, a)
        a_attended = self.cross_attn_a(a, v)

        # ✅ GATING fusion
        combined = torch.cat([v_attended, a_attended], dim=1)
        gates = self.gate(combined)

        fused = gates[:, 0:1] * v_attended + gates[:, 1:2] * a_attended

        return self.classifier(fused)
        # ⚠ workaround per mantenere classifier invariato
        #final_input = torch.cat([fused, fused], dim=1)

        #return self.classifier(final_input)
    

# ==========================================
# 3. SETUP TRAINING & EARLY STOPPING
# ==========================================
#weights = compute_class_weight(class_weight='balanced', classes=np.unique(y_encoded), y=y_encoded)

#class_weights_tensor = torch.tensor(weights, dtype=torch.float32)

k_folds = 5
skf = StratifiedKFold(n_splits=k_folds, shuffle=True, random_state=42)



MAX_EPOCHS = 100
PATIENCE = 3  






logger = logging.getLogger()
logger.setLevel(logging.INFO)

file_handler = logging.FileHandler("training_log.txt", mode="w")
file_handler.setLevel(logging.INFO)

console_handler = logging.StreamHandler(sys.stdout)
console_handler.setLevel(logging.INFO)

formatter = logging.Formatter("%(asctime)s - %(message)s")
file_handler.setFormatter(formatter)
console_handler.setFormatter(formatter)

logger.handlers = []  # evita duplicati
logger.addHandler(file_handler)
logger.addHandler(console_handler)

def log(msg):
    logger.info(msg)

results = []


#SHARED_DIM': 256,
#  'NUM_HEADS': 2,
#  'DROPOUT': 0.2,
#  'LR': 0.0001,

SHARED_DIMS = [256]
NUM_HEADS_LIST = [2, 4, 8]
DROPOUTS = [0.2, 0.3, 0.4]
LEARNING_RATES = [0.001, 3e-4, 3e-5]
all_preds, all_labels = [], []


for SHARED_DIM in SHARED_DIMS:
    for NUM_HEADS in NUM_HEADS_LIST:
        for LEARNING_RATE in LEARNING_RATES:
            for DROPOUT in DROPOUTS:

                fold_accuracies = []

                # ✅ AGGREGAZIONE GLOBALE PER F1
                all_preds, all_labels = [], []

                skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

                for fold, (train_idx, val_idx) in enumerate(skf.split(X_visual, y_encoded)):

                    X_v_train, X_v_val = X_visual[train_idx], X_visual[val_idx]
                    X_a_train, X_a_val = X_audio[train_idx], X_audio[val_idx]
                    y_train, y_val = y_encoded[train_idx], y_encoded[val_idx]

                    scaler_v = StandardScaler()
                    X_v_train_scaled = scaler_v.fit_transform(X_v_train)
                    X_v_val_scaled = scaler_v.transform(X_v_val)

                    scaler_a = StandardScaler()
                    X_a_train_scaled = scaler_a.fit_transform(X_a_train)
                    X_a_val_scaled = scaler_a.transform(X_a_val)

                    train_loader = DataLoader(TensorDataset(
                        torch.tensor(X_v_train_scaled, dtype=torch.float32),
                        torch.tensor(X_a_train_scaled, dtype=torch.float32),
                        torch.tensor(y_train, dtype=torch.long)
                    ), batch_size=32, shuffle=True)

                    val_loader = DataLoader(TensorDataset(
                        torch.tensor(X_v_val_scaled, dtype=torch.float32),
                        torch.tensor(X_a_val_scaled, dtype=torch.float32),
                        torch.tensor(y_val, dtype=torch.long)
                    ), batch_size=32, shuffle=False)

                    model = FusionWithCrossAttention(
                        visual_dim=X_visual.shape[1],
                        audio_dim=X_audio.shape[1],
                        SHARED_DIM=SHARED_DIM,
                        num_classes=len(le.classes_),
                        NUM_HEADS=NUM_HEADS,
                        DROPOUT=DROPOUT
                    )

                    class_weights_tensor = torch.tensor(
                        compute_class_weight(
                            class_weight='balanced',
                            classes=np.unique(y_encoded),
                            y=y_encoded
                        ),
                        dtype=torch.float32
                    )

                    criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)
                    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)

                    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
                        optimizer, mode='min', factor=0.5, patience=PATIENCE
                    )

                    best_val_loss = float('inf')
                    epochs_no_improve = 0

                    for epoch in range(MAX_EPOCHS):
                        model.train()

                        for inputs_v, inputs_a, labels in train_loader:

                            inputs_v = inputs_v + 0.01 * torch.randn_like(inputs_v)

                            optimizer.zero_grad()
                            outputs = model(inputs_v, inputs_a)
                            loss = criterion(outputs, labels)
                            loss.backward()
                            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                            optimizer.step()

                        model.eval()
                        val_loss = 0.0

                        with torch.no_grad():
                            for inputs_v, inputs_a, labels in val_loader:
                                outputs = model(inputs_v, inputs_a)
                                loss = criterion(outputs, labels)
                                val_loss += loss.item()

                        val_loss /= len(val_loader)
                        scheduler.step(val_loss)

                        if val_loss < best_val_loss:
                            best_val_loss = val_loss
                            epochs_no_improve = 0
                            best_model_state = model.state_dict()
                        else:
                            epochs_no_improve += 1

                        if epochs_no_improve >= PATIENCE:
                            break

                    model.load_state_dict(best_model_state)
                    model.eval()

                    preds, labels_list = [], []

                    with torch.no_grad():
                        for inputs_v, inputs_a, labels in val_loader:
                            outputs = model(inputs_v, inputs_a)
                            _, p = torch.max(outputs, 1)
                            preds.extend(p.numpy())
                            labels_list.extend(labels.numpy())

                    acc = accuracy_score(labels_list, preds)
                    fold_accuracies.append(acc)

                    # ✅ AGGREGA PER F1 GLOBALE
                    all_preds.extend(preds)
                    all_labels.extend(labels_list)

                mean_acc = np.mean(fold_accuracies)

                # ✅ F1 CORRETTO SU TUTTI I FOLD
                f1 = f1_score(all_labels, all_preds, average='macro')

                results.append({
                    "SHARED_DIM": SHARED_DIM,
                    "NUM_HEADS": NUM_HEADS,
                    "DROPOUT": DROPOUT,
                    "LR": LEARNING_RATE,
                    "ACC": mean_acc,
                    "F1": f1
                })

                log(f"Tested Config: SHARED_DIM={SHARED_DIM}, NUM_HEADS={NUM_HEADS}, DROPOUT={DROPOUT}, LR={LEARNING_RATE} -> MEAN ACC: {mean_acc:.4f}, F1: {f1:.4f}")



RuntimeError: mat1 and mat2 shapes cannot be multiplied (32x512 and 256x2)

In [ ]:
#max(results, key=lambda x: x['ACC'])
sorted(results, key=lambda x: x["ACC"], reverse=True)[:5]

###### {'SHARED_DIM': 256,
######   'NUM_HEADS': 2,
######   'DROPOUT': 0.2,
######   'LR': 0.0001,
######   'ACC': np.float64(0.6436204744069913)}